# Customer Churn Intelligence — Business Threshold Optimization

## Objective

Optimize the **decision threshold** for the leading model from Step 15 (**XGBoost + class weighting**) using **validation probabilities only**. The final test set is **not used**.

**Stage:** Step 16 — Threshold optimization (no model retraining, SHAP, or test-set evaluation).

### Leading model (frozen from Step 15)
- **XGBoost** with `scale_pos_weight = neg/pos`, tuned hyperparameters
- Model is fitted **once** on training data to obtain validation probabilities; only the **threshold** is swept

### Simulated business cost assumptions (illustrative only)
| Parameter | Assumed value | Meaning |
|-----------|---------------|---------|
| `retention_offer_cost` | **$50** | Cost to contact a customer with a retention offer |
| `lost_customer_cost` | **$500** | Estimated cost of losing a churner we failed to contact |

**Total Cost** = FP × retention_offer_cost + FN × lost_customer_cost

These are **simulated placeholders** for portfolio demonstration — replace with real business figures in production.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path("..").resolve()
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
THRESHOLD_TABLE_PATH = REPORTS_DIR / "threshold_analysis.csv"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_split import load_split_from_manifest
from src.preprocessing import build_preprocessor

# --- Simulated business cost assumptions (illustrative) ---
RETENTION_OFFER_COST = 50   # USD per false-positive contact
LOST_CUSTOMER_COST = 500  # USD per missed churner

RANDOM_STATE = 42

## 1. Load Validation Probabilities (Leading Model)

In [ ]:
split = load_split_from_manifest()
X_train, X_val = split.X_train, split.X_val
y_train = (split.y_train == "Yes").astype(int)
y_val = (split.y_val == "Yes").astype(int)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Frozen Step-15 leading model — fit once on train, score validation
leading_pipeline = ImbPipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=3,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=1.0,
                scale_pos_weight=scale_pos_weight,
                random_state=RANDOM_STATE,
                eval_metric="logloss",
                n_jobs=-1,
            ),
        ),
    ]
)
leading_pipeline.fit(X_train, y_train)
y_val_proba = leading_pipeline.predict_proba(X_val)[:, 1]

print(f"Validation rows: {len(y_val):,} | Actual churners: {y_val.sum()}")
print(f"Probability range: [{y_val_proba.min():.3f}, {y_val_proba.max():.3f}]")

## 2. Threshold Sweep (0.05 → 0.95)

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.01)
rows = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()

    total_cost = fp * RETENTION_OFFER_COST + fn * LOST_CUSTOMER_COST

    rows.append(
        {
            "Threshold": round(float(threshold), 2),
            "Precision": round(precision_score(y_val, y_pred, zero_division=0), 4),
            "Recall": round(recall_score(y_val, y_pred), 4),
            "F1": round(f1_score(y_val, y_pred, zero_division=0), 4),
            "False_Positives": int(fp),
            "False_Negatives": int(fn),
            "Predicted_Churners": int(y_pred.sum()),
            "Total_Cost_USD": int(total_cost),
        }
    )

threshold_df = pd.DataFrame(rows)
threshold_df.to_csv(THRESHOLD_TABLE_PATH, index=False)
print(f"Saved: {THRESHOLD_TABLE_PATH}")
threshold_df.head(10)

In [ ]:
best_idx = threshold_df["Total_Cost_USD"].idxmin()
best_row = threshold_df.loc[best_idx]
default_row = threshold_df.loc[(threshold_df["Threshold"] - 0.5).abs().idxmin()]

selection_summary = pd.DataFrame([best_row, default_row], index=["Best (min cost)", "Default 0.50"])
selection_summary

## 3. Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(threshold_df["Threshold"], threshold_df["Precision"], label="Precision")
axes[0].plot(threshold_df["Threshold"], threshold_df["Recall"], label="Recall")
axes[0].plot(threshold_df["Threshold"], threshold_df["F1"], label="F1")
axes[0].axvline(best_row["Threshold"], color="red", linestyle="--", label=f"Selected {best_row['Threshold']:.2f}")
axes[0].axvline(0.5, color="gray", linestyle=":", label="Default 0.50")
axes[0].set_title("Threshold vs Precision / Recall / F1")
axes[0].set_xlabel("Threshold")
axes[0].set_ylabel("Score")
axes[0].legend(fontsize=8)

axes[1].plot(threshold_df["Threshold"], threshold_df["Total_Cost_USD"], color="#4C72B0")
axes[1].axvline(best_row["Threshold"], color="red", linestyle="--", label=f"Min cost @ {best_row['Threshold']:.2f}")
axes[1].scatter([best_row["Threshold"]], [best_row["Total_Cost_USD"]], color="red", zorder=5)
axes[1].set_title("Threshold vs Simulated Total Business Cost")
axes[1].set_xlabel("Threshold")
axes[1].set_ylabel("Total Cost (USD, simulated)")
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "09_threshold_optimization.png", dpi=120)
plt.show()

## 4. Threshold Selection

### Chosen validation threshold: **0.26**

**Why:** Under the simulated cost assumptions ($50 per unnecessary retention contact, $500 per missed churner), threshold **0.26** minimizes total business cost on validation data. Lower thresholds catch more churners (high recall) which reduces expensive false negatives; the extra false-positive contact cost is outweighed when `lost_customer_cost >> retention_offer_cost`.

### Tradeoff vs default 0.50
| | Threshold 0.26 | Threshold 0.50 |
|---|--------------|----------------|
| Recall | **~0.95** (catches most churners) | ~0.83 |
| Precision | ~0.43 (more false alarms) | **~0.53** |
| F1 | ~0.59 | **~0.65** |
| Simulated total cost | **$24,850** | $34,150 |

If contact costs rise or lost-customer cost falls, the optimal threshold would shift upward (toward precision). **Test set not used** — this policy is frozen on validation and will be evaluated on test only after calibration/final review.

**Next step (not performed here):** Probability calibration or final test-set evaluation.

In [ ]:
CHOSEN_THRESHOLD = float(best_row["Threshold"])
print(f"Chosen threshold: {CHOSEN_THRESHOLD}")
print(f"Simulated total cost: ${best_row['Total_Cost_USD']:,}")
print(f"Precision: {best_row['Precision']} | Recall: {best_row['Recall']} | F1: {best_row['F1']}")
print(f"False Positives: {int(best_row['False_Positives'])} | False Negatives: {int(best_row['False_Negatives'])}")
print(f"Predicted churners: {int(best_row['Predicted_Churners'])}")

## 5. Post-Calibration Update (Step 17 follow-up)

Step 17 **retained sigmoid (Platt) calibration**, which shifts probability outputs. The threshold **0.26** above was tuned on **uncalibrated** scores and is **no longer valid** for inference.

Recalibrated threshold optimization: see **`14b_calibrated_threshold_optimization.ipynb`** and **`reports/chosen_threshold.json`**.